In [61]:
!pip install datasets rank_bm25 sentence-transformers spacy faiss-cpu transformers==4.46.3 keybert==0.8.5

In [62]:
from datasets import load_dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split = 'train')
print(dataset)

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [63]:
import pandas as pd
df = pd.DataFrame(dataset)
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [64]:
df = df[['title', 'abstract']]
print("Original shape of dataset:", df.shape)

Original shape of dataset: (117592, 2)


In [65]:
df = df.sample(15000, random_state=42)
print("Shape after sampling:", df.shape)

Shape after sampling: (15000, 2)


In [67]:
from rank_bm25 import BM25Okapi
df["paper_text"] = df["title"] + " " + df["abstract"]
df["fast_text"] = df["title"] + " " + df["abstract"]

tokenized_corpus = [doc.split() for doc in df["paper_text"]]
bm25 = BM25Okapi(tokenized_corpus)

In [69]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [70]:
print(df.columns)

Index(['title', 'abstract', 'paper_text', 'fast_text'], dtype='object')


In [71]:
print(df["paper_text"].iloc[0])
print(type(df["paper_text"].iloc[0]))

SVD Perspectives for Augmenting DeepONet Flexibility and
  Interpretability   Deep operator networks (DeepONets) are powerful architectures for fast and
accurate emulation of complex dynamics. As their remarkable generalization
capabilities are primarily enabled by their projection-based attribute, we
investigate connections with low-rank techniques derived from the singular
value decomposition (SVD). We demonstrate that some of the concepts behind
proper orthogonal decomposition (POD)-neural networks can improve DeepONet's
design and training phases. These ideas lead us to a methodology extension that
we name SVD-DeepONet. Moreover, through multiple SVD analyses, we find that
DeepONet inherits from its projection-based attribute strong inefficiencies in
representing dynamics characterized by symmetries. Inspired by the work on
shifted-POD, we develop flexDeepONet, an architecture enhancement that relies
on a pre-transformation network for generating a moving reference frame and
isolat

In [72]:
from sentence_transformers import SentenceTransformer

In [73]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [74]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [75]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex = False)
df["paper_text"] = df["paper_text"].str.strip()

NER Setup

In [76]:
import spacy

nlp = spacy.load("en_core_web_sm")
print("spaCy loaded successfully")

spaCy loaded successfully


In [77]:
def extract_entities_batch(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=50):
        ents = list(set([ent.text.lower() for ent in doc.ents]))
        results.append(ents)
    return results

df["entities"] = extract_entities_batch(df["paper_text"].tolist())

In [78]:
df["entities_text"] = df["entities"].apply(lambda x: " ".join(x))
df["enhanced_text"] = df["paper_text"] + " " + df["entities_text"]
df["enhanced_text"] = df["enhanced_text"].str[:1200]

In [79]:
sample_text = df["enhanced_text"].iloc[0]
embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [80]:
embedding[:56]

array([-0.07784659, -0.0508581 ,  0.01979457,  0.00296188, -0.00117054,
        0.03070743, -0.09895527, -0.03836545,  0.07000241, -0.06716654,
       -0.01217733, -0.00648127, -0.12996532,  0.04775481, -0.0918047 ,
       -0.02913623,  0.0552972 ,  0.15252027, -0.13286482, -0.01332718,
        0.02445521, -0.00098864,  0.02860047,  0.00163133,  0.09225886,
        0.03326268, -0.03570014, -0.02493018,  0.0173664 , -0.05142784,
        0.05207576, -0.02710167, -0.09572868, -0.02476717, -0.06918894,
        0.11361069,  0.03494626,  0.07869286, -0.09358688, -0.0319358 ,
        0.03792727,  0.08685069,  0.0339055 ,  0.0593663 ,  0.02664454,
        0.04765074,  0.00584963, -0.11035454, -0.00769774,  0.06845919,
       -0.0582855 ,  0.02893853,  0.05209697,  0.05537824,  0.02491188,
       -0.01971435], dtype=float32)

In [81]:
sample_embedding = model.encode(df["enhanced_text"].head(5).to_list())
print(sample_embedding.shape)

(5, 384)


In [82]:
from sklearn.metrics.pairwise import cosine_similarity
for i in range(1, 5):
  sim = cosine_similarity(sample_embedding[0].reshape(1, -1),sample_embedding[i].reshape(1, -1))
  print(f"Similarity with doc {i}: {sim[0][0]:.4f}")

Similarity with doc 1: 0.2577
Similarity with doc 2: 0.2097
Similarity with doc 3: 0.0778
Similarity with doc 4: 0.2956


Generate Full Embedding

In [83]:
import os
import numpy as np

if os.path.exists("paper_embeddings.npy"):
  print("Loading saved embeddings")
  embeddings = np.load("paper_embeddings.npy")
else:
  print("Generating embeddings")
  embeddings = model.encode(
      df["fast_text"].tolist(),
      batch_size=32,
      show_progress_bar=True
  )

  embeddings = embeddings.astype("float32")
  np.save("paper_embeddings.npy", embeddings)
  print("Embeddings saved successfully!")

Loading saved embeddings


In [84]:
print("Shape:", embeddings.shape)
print(type(embeddings))
print("Data Type:", embeddings.dtype)

Shape: (15000, 384)
<class 'numpy.ndarray'>
Data Type: float32


In [85]:
import faiss

if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")

    embeddings = embeddings.astype("float32")
    faiss.normalize_L2(embeddings)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)

    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")

    print("FAISS index saved successfully!")

Loading existing FAISS index


In [86]:
print("Total vectors in index:", index.ntotal)

Total vectors in index: 15000


In [87]:
query = "deep learning for medical image analysis"
query_embedding = model.encode([query])
query_embedding = query_embedding.astype("float32")
print(query_embedding.shape)

(1, 384)


In [88]:
faiss.normalize_L2(query_embedding)
D, I = index.search(query_embedding, 5)
print("Top indices:", I)
print("Top scores:", D)

Top indices: [[14303  9768   985  4213  9754]]
Top scores: [[0.7254224  0.66002    0.64228296 0.63344663 0.6283094 ]]


In [89]:
def search_paper(query, k=5, top_k=20):
    query_embedding = model.encode([query])
    query_embedding = query_embedding.astype("float32")
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, top_k)

    pairs = [(query, df.iloc[idx]["enhanced_text"]) for idx in I[0]]
    rerank_scores = reranker.predict(pairs)

    reranked = sorted(zip(rerank_scores, I[0]), key=lambda x: x[0], reverse=True)

    for score, idx in reranked[:k]:
      faiss_score = D[0][list(I[0]).index(idx)]

      print("="*80)
      print(f"Rerank score: {score}")
      print(f"FAISS score: {faiss_score:.4f}")
      print(f"Title: {df.iloc[idx]['title']}")
      print()
      print(f"Abstract: {df.iloc[idx]['abstract'][:500]}")
      print("="*80)
      print()

In [90]:
search_paper("deep learning for medical image analysis")

Rerank score: 10.199779510498047
FAISS score: 0.6600
Title: A Tour of Unsupervised Deep Learning for Medical Image Analysis

Abstract:   Interpretation of medical images for diagnosis and treatment of complex
disease from high-dimensional and heterogeneous data remains a key challenge in
transforming healthcare. In the last few years, both supervised and
unsupervised deep learning achieved promising results in the area of medical
imaging and image analysis. Unlike supervised learning which is biased towards
how it is being supervised and manual efforts to create class label for the
algorithm, unsupervised learning derive insigh

Rerank score: 9.59546184539795
FAISS score: 0.5947
Title: Deep Learning for Automated Medical Image Analysis

Abstract:   Medical imaging is an essential tool in many areas of medical applications,
used for both diagnosis and treatment. However, reading medical images and
making diagnosis or treatment recommendations require specially trained medical
specialist

In [91]:
from transformers import pipeline
summarizer = pipeline(
    task = "summarization",
    model = "facebook/bart-large-cnn"
)

In [92]:
from keybert import KeyBERT
kw_model = KeyBERT(model)
type(kw_model)

keybert._model.KeyBERT

In [93]:
summary = summarizer(df.iloc[14303]["abstract"], max_length=120, min_length=40)
print(summary[0]["summary_text"])

Deep neural networks are being used to improve the quality of medical images. The technology is being used in a variety of fields, including medicine and education. It is also being used as a tool to help people understand and understand their health.


In [94]:
def expand_query(query):
    keywords = kw_model.extract_keywords(query, top_n=3)
    expanded = query + " " + " ".join([k for k, _ in keywords])
    return expanded

In [95]:
expanded_query = expand_query(query)

In [96]:
def search_and_summarize(query, k=5, top_k=20):
    expanded_query = expand_query(query)

    query_embedding = model.encode([expanded_query]).astype("float32")
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, top_k)

    tokenized_query = expanded_query.split()
    bm25_scores = bm25.get_scores(tokenized_query)
    max_bm25 = max(bm25_scores) if max(bm25_scores) != 0 else 1

    candidates = I[0]

    pairs = [(expanded_query, df.iloc[idx]["enhanced_text"]) for idx in candidates]

    rerank_scores = reranker.predict(pairs)
    rerank_scores = 1 / (1 + np.exp(-rerank_scores))

    final_results = []

    for i, idx in enumerate(candidates):
        faiss_score = D[0][i]
        bm25_norm = bm25_scores[idx] / max_bm25
        rerank = rerank_scores[i]

        final_score = 0.6 * rerank + 0.3 * faiss_score + 0.1 * bm25_norm
        final_results.append((final_score, idx))

    final_results.sort(reverse=True)

    for score, idx in final_results[:k]:
        print("=" * 80)
        print(f"Final score: {score:.4f}")
        print(f"Title: {df.iloc[idx]['title']}")
        print()

        abstract = df.iloc[idx]["abstract"]
        print(f"Abstract: {abstract[:500]}")
        print()

        try:
            summary = summarizer(
                abstract[:1024],
                max_length=120,
                min_length=40,
                do_sample=False
            )
            print(f"Summary: {summary[0]['summary_text']}")
        except:
            print("Summary error")

        print()

        try:
            keywords = kw_model.extract_keywords(
                abstract,
                keyphrase_ngram_range=(1, 3),
                stop_words="english",
                top_n=5
            )

            print("Keywords:")
            for kw, _ in keywords:
                print(f"- {kw}")
        except:
            print("Keyword error")

        print("=" * 80)
        print()

In [97]:
search_and_summarize("deep learning for medical image analysis", k=5)

Final score: 0.9168
Title: An overview of deep learning in medical imaging focusing on MRI

Abstract:   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural l

Summary: Machine learning has seen a tremendous amount of attention over the last few years. Deep neural networks are now the state-of-the-art machine learning models. These developments have a huge potential for medical technology, data analysis, medical diagnostics and healthcare.

Keywords:
- deep learning mri
- learning mri
- introduction deep learning
- learning medical imaging
- deep neu

In [98]:
df.to_csv("papers.csv", index=False)

In [99]:
import pickle

with open("bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)